In [35]:
import pandas as pd
import numpy as np

In [3]:


# Load the parquet file
df = pd.read_parquet("gdelt_semiconductor_events.parquet")

# Show the first 5 rows (use df.head(6) for 6)
print(df.head(5))

  GLOBALEVENTID   SQLDATE       DATEADDED Actor1Name Actor1CountryCode  \
0    1149157629  20240101  20240101000000       None              None   
1    1149157960  20240101  20240101000000     GERMAN               DEU   
2    1149157961  20240101  20240101000000     GERMAN               DEU   
3    1149157969  20240101  20240101000000     GERMAN               DEU   
4    1149157972  20240101  20240101000000     GERMAN               DEU   

  Actor2Name Actor2CountryCode ActionGeo_CountryCode  \
0     ISRAEL               ISR                    IS   
1    ISLAMIC              None                    GM   
2    ISLAMIC              None                    US   
3    COLOGNE               DEU                    GM   
4    ISLAMIC              None                    GM   

                      ActionGeo_FullName EventCode EventBaseCode  \
0         Gaza, Israel (general), Israel       141           141   
1                Berlin, Berlin, Germany       173           173   
2  Times Squar

In [4]:
df = pd.read_parquet("ukraine_russia_critical_events.parquet")

# Show the first 5 rows (use df.head(6) for 6)
print(df.head(5))

  GLOBALEVENTID   SQLDATE       DATEADDED Actor1Name Actor1CountryCode  \
0    1149158665  20240101  20240101000000  UKRAINIAN               UKR   
1    1149191247  20240101  20240101081500    RUSSIAN               RUS   
2    1149325529  20240102  20240102114500  UKRAINIAN               UKR   
3    1149363223  20240102  20240102163000   MILITARY              None   
4    1149363225  20240102  20240102163000   MILITARY              None   

  Actor2Name Actor2CountryCode ActionGeo_CountryCode  \
0   BELGOROD               RUS                    RS   
1  UKRAINIAN               UKR                    RS   
2     RUSSIA               RUS                    RS   
3    UKRAINE               UKR                    UP   
4  UKRAINIAN               UKR                    UP   

                        ActionGeo_FullName EventCode  ... EventRootCode  \
0  Belgorod, Belgorodskaya Oblast', Russia       190  ...            19   
1                                   Russia       193  ...           

In [5]:
def gdelt_to_country_month(ev, country_col="Actor1CountryCode"):
    """Collapse raw GDELT events -> one row per country per month."""
    ev = ev.copy()
    ev["ym"] = pd.to_datetime(ev["date"]).dt.to_period("M")
    ev = ev.dropna(subset=[country_col])
    # coverage-weighted intensity: loudly-reported conflict counts more
    ev["intensity"] = ev["GoldsteinScale"].abs() * ev["NumMentions"]
    base = ev.groupby([country_col, "ym"]).agg(
        n_events  =("GLOBALEVENTID", "size"),
        intensity =("intensity", "sum"),
        goldstein =("GoldsteinScale", "mean"),
        tone      =("AvgTone", "mean"),
    )
    cls = (ev.groupby([country_col, "ym", "EventRootCode"]).size()
             .unstack("EventRootCode", fill_value=0))
    cls.columns = [f"n_root_{c}" for c in cls.columns]
    return base.join(cls).reset_index().rename(columns={country_col: "iso3"})

def monthly_to_annual(cm):
    """Roll the monthly table up into rich per-country-per-year features."""
    cm = cm.copy()
    cm["year"] = cm["ym"].dt.year
    return cm.groupby(["iso3", "year"]).agg(
        events_total  =("n_events", "sum"),
        intensity_sum =("intensity", "sum"),
        intensity_max =("intensity", "max"),   # worst month = the shock
        intensity_std =("intensity", "std"),   # volatility across the year
        goldstein_min =("goldstein", "min"),   # most conflictual month
        tone_min      =("tone", "min"),
    )

In [6]:
ev = pd.read_parquet("gdelt_semiconductor_events.parquet")
cm = gdelt_to_country_month(ev)     # country × month
cy = monthly_to_annual(cm)          # country × year
print(cm.head())
print(cy.head())

  iso3       ym  n_events  intensity  goldstein      tone  n_root_14  \
0  AFR  2024-01         1      180.0  -9.000000 -7.425265          0   
1  ARE  2024-01         5      480.0 -10.000000 -7.593123          0   
2  AUS  2024-01         9     3480.0  -9.444444 -8.876968          0   
3  AUT  2024-01         2       80.0  -5.000000 -0.476425          0   
4  CAN  2024-01         2      200.0 -10.000000 -1.410224          0   

   n_root_17  n_root_18  n_root_19  n_root_20  
0          0          1          0          0  
1          0          0          5          0  
2          1          0          8          0  
3          2          0          0          0  
4          0          0          2          0  
           events_total  intensity_sum  intensity_max  intensity_std  \
iso3 year                                                              
AFR  2024             1          180.0          180.0            NaN   
ARE  2024             5          480.0          480.0          

In [7]:
ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")
ev = ev[pd.to_datetime(ev["date"]).dt.year.between(2017, 2023)]
cm = gdelt_to_country_month(ev)
print(sorted(cm["ym"].dt.year.unique()))   # want [2017,...,2023], not just [2024]

[2017, 2018, 2019, 2020, 2021, 2022, 2023]


In [8]:
def events_in(ev, country, year=None, month=None,
              country_field="any", sort_by="intensity", top=None):
    """
    Return the raw events for a country (and optional year/month).

    country       : ISO3 code, e.g. "CHN", "TWN", "KOR"
    year          : e.g. 2022   (None = all years)
    month         : 1–12        (None = whole year)
    country_field : "any" (actor1 OR actor2 OR action-geo), or a specific
                    column name like "Actor1CountryCode"
    sort_by       : "intensity" (|Goldstein|×Mentions), "GoldsteinScale",
                    "NumMentions", or "date"
    top           : keep only the N highest-ranked (None = all)
    """
    d = ev.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")

    # --- country filter ---
    if country_field == "any":
        mask = (d["Actor1CountryCode"] == country) | \
               (d["Actor2CountryCode"] == country) | \
               (d["ActionGeo_CountryCode"] == country)
    else:
        mask = d[country_field] == country
    d = d[mask]

    # --- time filter ---
    if year is not None:
        d = d[d["date"].dt.year == year]
    if month is not None:
        d = d[d["date"].dt.month == month]

    if d.empty:
        print(f"No events for {country} {year or ''}-{month or ''}")
        return d

    # --- scoring + sort ---
    for c in ["GoldsteinScale", "NumMentions"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["intensity"] = d["GoldsteinScale"].abs() * d["NumMentions"]
    d = d.sort_values(sort_by if sort_by != "date" else "date",
                      ascending=(sort_by == "date"))

    if top:
        d = d.head(top)
    return d

In [9]:
ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")

# everything Taiwan-related in 2022, ranked by intensity:
tay22=events_in(ev, "TWN", 2022)[
    ["date","Actor1Name","Actor2Name","event_class",
     "EventRootCode","GoldsteinScale","NumMentions","intensity","SOURCEURL"]
].head(20)

tay22

,date,Actor1Name,Actor2Name,event_class,EventRootCode,GoldsteinScale,NumMentions,intensity,SOURCEURL
3180009,2022-05-16,GENEVA,TAIWANESE,small_arms_conflict,19,-10.0,2846,28460.0,https://www.washingtonpost.com/national/author...
3269988,2022-07-31,None,TAIWAN,aerial_weapons,19,-10.0,2052,20520.0,https://www.wdiy.org/npr-news/npr-news/2022-07...
3269755,2022-07-30,None,TAIWAN,aerial_weapons,19,-10.0,1680,16800.0,https://www.sfgate.com/news/article/China-anno...
3276824,2022-08-05,MILITARY,TAIWAN,artillery_conflict,19,-10.0,1530,15300.0,http://www.msn.com/en-us/news/world/china-halt...
3279462,2022-08-07,CHINA,TAIWAN,military_force,19,-10.0,1420,14200.0,https://www.sfgate.com/news/article/China-keep...
3371222,2022-11-12,TAIWAN,None,artillery_conflict,19,-10.0,1188,11880.0,https://www.startribune.com/chinas-xi-out-of-c...
3277623,2022-08-06,MILITARY,TAIWAN,artillery_conflict,19,-10.0,1000,10000.0,https://www.maldonandburnhamstandard.co.uk/new...
3282018,2022-08-09,MILITARY,PINGTUNG COUNTY,artillery_conflict,19,-10.0,856,8560.0,https://www.middletownpress.com/news/article/T...
3216266,2022-06-12,TAIWAN,CHINA,military_force,19,-10.0,852,8520.0,https://www.wbal.com/article/570282/127/china-...
3283497,2022-08-10,MILITARY,PINGTUNG COUNTY,artillery_conflict,19,-10.0,818,8180.0,https://www.thesunchronicle.com/news/nation_wo...


In [10]:
# the 10 biggest China events in October 2022 (export-controls month):

china_oc22=events_in(ev, "CHN", 2022, month=10, top=10)
china_oc22

,GLOBALEVENTID,SQLDATE,DATEADDED,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_CountryCode,ActionGeo_FullName,EventCode,...,EventRootCode,event_class,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL,date,intensity
3347946,1067773214,20221016,20221016,BEIJING,CHN,CONGRESS,None,CH,"Beijing, Beijing, China",173,...,17,arrest_or_detain,-5.0,1618,116,1618,-0.304677,https://www.clactonandfrintongazette.co.uk/new...,2022-10-16,8090.0
3357499,1069241455,20221026,20221026,CHINESE,CHN,COMPANY,None,CH,"Beijing, Beijing, China",193,...,19,small_arms_conflict,-10.0,562,46,562,-0.892355,https://www.wfmz.com/news/china-launches-a-cov...,2022-10-26,5620.0
3350982,1068259664,20221019,20221019,POLICE,None,CHINESE,CHN,CH,China,180,...,18,assault_security_risk,-9.0,548,68,548,-2.581026,https://www.eveshamjournal.co.uk/news/national...,2022-10-19,4932.0
3333339,1065686475,20221002,20221002,CHINESE,CHN,None,None,US,"Minnesota, United States",1821,...,18,assault_security_risk,-9.0,545,69,545,-8.118658,https://www.dailyherald.com/article/20221002/n...,2022-10-02,4905.0
3352900,1068561544,20221021,20221021,UNITED KINGDOM,GBR,CHINA,CHN,CH,"Beijing, Beijing, China",175,...,17,repression,-9.0,540,54,540,-6.815376,https://www.argentinastar.com/news/272962279/c...,2022-10-21,4860.0
3333302,1065679793,20221002,20221002,CHINESE,CHN,None,None,US,"Minneapolis, Minnesota, United States",1821,...,18,assault_security_risk,-9.0,538,82,538,-7.425182,https://mynorthwest.com/3657293/chinese-billio...,2022-10-02,4842.0
3357498,1069241454,20221026,20221026,CHINESE,CHN,COMPANY,None,CH,"Shanghai, Shanghai, China",193,...,19,small_arms_conflict,-10.0,482,47,482,-0.238943,https://economictimes.indiatimes.com/news/inte...,2022-10-26,4820.0
3352891,1068559558,20221021,20221021,None,None,SHANGHAI,CHN,CH,"Shanghai, Shanghai, China",141,...,14,civil_unrest,-6.5,660,64,660,-3.071584,https://www.goskagit.com/news/world/as-leaders...,2022-10-21,4290.0
3345236,1067406273,20221013,20221013,CHINA,CHN,None,None,CH,"Beijing, Beijing, China",191,...,19,blockade,-9.5,440,44,440,-5.400391,https://www.klcc.org/npr-world-news/2022-10-13...,2022-10-13,4180.0
3357510,1069244563,20221026,20221026,CHINESE,CHN,COMPANY,None,CH,"Beijing, Beijing, China",193,...,19,small_arms_conflict,-10.0,376,35,376,-1.296589,https://www.whec.com/national-world/china-laun...,2022-10-26,3760.0


In [11]:
usa=events_in(ev,"USA",2017)
usa

,GLOBALEVENTID,SQLDATE,DATEADDED,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_CountryCode,ActionGeo_FullName,EventCode,...,EventRootCode,event_class,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL,date,intensity
586765,704165397,20171106,20171106,TEXAS,USA,None,None,US,"Texas, United States",193,...,19,small_arms_conflict,-10.0,28455,1627,23414,-6.084484,http://www.geelongadvertiser.com.au/news/world...,2017-11-06,284550.0
556872,694335673,20171002,20171002,UNITED STATES,USA,None,None,US,"Las Vegas, Nevada, United States",193,...,19,small_arms_conflict,-10.0,24132,1467,19495,-6.032438,http://www.lasvegasnow.com/news/update-child-f...,2017-10-02,241320.0
559721,694589228,20171003,20171003,NASHVILLE,USA,None,None,US,"Las Vegas, Nevada, United States",193,...,19,small_arms_conflict,-10.0,21829,1111,16180,-4.843193,http://www.850wftl.com/watch-vigil-held-las-ve...,2017-10-03,218290.0
38288,618863104,20170120,20170120,UNITED STATES,USA,None,None,US,"Washington, District of Columbia, United States",190,...,19,military_force,-10.0,18882,704,18814,-4.806829,http://www.cbc.ca/news/world/trump-inauguratio...,2017-01-20,188820.0
590104,704469120,20171107,20171107,TEXAS,USA,None,None,US,"Texas, United States",193,...,19,small_arms_conflict,-10.0,15144,1109,13523,-5.739935,http://www.kltv.com/story/36771707/the-latest-...,2017-11-07,151440.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52398,620942559,20170126,20170126,UNITED STATES,USA,UNITED STATES,USA,US,"Hollywood, California, United States",173,...,17,arrest_or_detain,-5.0,5,2,5,-3.696566,http://www.wbtv.com/story/34350504/actor-shia-...,2017-01-26,25.0
40908,619211459,20170121,20170121,MEXICO,MEX,UNITED STATES,USA,US,"Brooklyn, Illinois, United States",173,...,17,arrest_or_detain,-5.0,5,5,5,-7.053942,http://www.localsyr.com/news/el-chapo-guzman-p...,2017-01-21,25.0
593975,704859525,20171108,20171108,None,None,JERSEY,USA,JE,Jersey,173,...,17,arrest_or_detain,-5.0,5,3,5,-8.035870,http://www.njherald.com/20171107/three-face-dr...,2017-11-08,25.0
601829,706095672,20171112,20171112,UNITED STATES,USA,DANE,DNK,US,"Dane County, Wisconsin, United States",173,...,17,arrest_or_detain,-5.0,5,2,5,-1.862846,http://www.rivernewsonline.com/main.asp?Sectio...,2017-11-12,25.0


In [12]:
#second version :
def _as_set(v):
    """None -> no filter; scalar -> {scalar}; list/range -> set."""
    if v is None:
        return None
    if isinstance(v, (list, tuple, set, range)):
        return set(v)
    return {v}

def events_in(ev, country=None, year=None, month=None,
              country_field="any", sort_by="intensity", top=None):
    d = ev.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")

    # country (None = all)
    if country is not None:
        countries = _as_set(country)
        if country_field == "any":
            mask = (d["Actor1CountryCode"].isin(countries) |
                    d["Actor2CountryCode"].isin(countries) |
                    d["ActionGeo_CountryCode"].isin(countries))
        else:
            mask = d[country_field].isin(countries)
        d = d[mask]

    # year / month (None = all; scalar or list both work)
    years, months = _as_set(year), _as_set(month)
    if years is not None:
        d = d[d["date"].dt.year.isin(years)]
    if months is not None:
        d = d[d["date"].dt.month.isin(months)]

    if d.empty:
        print("No events for that selection"); return d

    for c in ["GoldsteinScale", "NumMentions"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["intensity"] = d["GoldsteinScale"].abs() * d["NumMentions"]
    d = d.sort_values(sort_by, ascending=(sort_by == "date"))
    return d.head(top) if top else d

In [13]:
print(ev["date"].dtype)
print(ev["date"].head())

datetime64[ns]
0   2016-01-02
1   2016-01-02
2   2017-01-01
3   2017-01-01
4   2017-01-01
Name: date, dtype: datetime64[ns]


In [14]:
events_in(ev, "TWN", year=[2018, 2019, 2020])

,GLOBALEVENTID,SQLDATE,DATEADDED,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_CountryCode,ActionGeo_FullName,EventCode,...,EventRootCode,event_class,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL,date,intensity
2213818,913045734,20200318,20200318,TAIWAN,TWN,None,None,TW,Taiwan,172,...,17,sanctions_or_admin_restrictions,-5.0,6309,516,6264,-4.564953,https://www.msn.com/en-in/finance/topstories/t...,2020-03-18,31545.0
1282320,796296188,20181021,20181021,None,None,TAIPEI,TWN,CH,"Beijing, Beijing, China",190,...,19,military_force,-10.0,1598,153,1598,-4.123840,http://www.wistv.com/2018/10/21/taiwan-train-d...,2018-10-21,15980.0
1282321,796296189,20181021,20181021,None,None,TAIWAN,TWN,TW,"Luodong, Yilan Xian, Taiwan",190,...,19,military_force,-10.0,1282,145,1282,-4.300501,https://www.ctvnews.ca/world/taiwan-train-dera...,2018-10-21,12820.0
2549642,955773302,20201122,20201122,TAIWAN,TWN,None,None,TW,"Taipei, T'ai-pei, Taiwan",141,...,14,civil_unrest,-6.5,1918,133,1638,-0.673076,https://abcnews.go.com/US/wireStory/thousands-...,2020-11-22,12467.0
1222666,788746631,20180921,20180921,TAIWAN,TWN,None,None,TW,"Taipei, T'ai-pei, Taiwan",173,...,17,arrest_or_detain,-5.0,2352,269,2350,-3.045276,https://www.pv-tech.org/news/taiwan-solar-2.0-...,2018-09-21,11760.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
980326,758597565,20180524,20180524,OPERATIVE,None,TAIWAN,TWN,TW,Taiwan,173,...,17,arrest_or_detain,-5.0,5,2,5,-6.640404,http://www.gmanetwork.com/news/news/nation/654...,2018-05-24,25.0
2150472,904630105,20200208,20200208,TAIWAN,TWN,CHINESE,CHN,TW,"Taipei, T'ai-pei, Taiwan",172,...,17,sanctions_or_admin_restrictions,-5.0,5,3,5,-2.723164,https://mainichi.jp/english/articles/20200208/...,2020-02-08,25.0
2150473,904630106,20200208,20200208,TAIWAN,TWN,CHINESE,CHN,CH,China,172,...,17,sanctions_or_admin_restrictions,-5.0,5,3,5,-2.723164,https://mainichi.jp/english/articles/20200208/...,2020-02-08,25.0
2417018,938632415,20200803,20200803,TAIPEI,TWN,None,None,TW,"Taipei, T'ai-pei, Taiwan",172,...,17,sanctions_or_admin_restrictions,-5.0,5,2,5,-2.623308,https://www.asiaone.com/china/taiwanese-airlin...,2020-08-03,25.0


In [15]:
events_in(ev, ["TWN","KOR","JPN"], year=2019)

,GLOBALEVENTID,SQLDATE,DATEADDED,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_CountryCode,ActionGeo_FullName,EventCode,...,EventRootCode,event_class,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL,date,intensity
1977700,883400150,20191031,20191031,None,None,JAPANESE,JPN,JA,"Okinawa, Okinawa, Japan",190,...,19,military_force,-10.0,3982,387,3922,-3.289793,https://www.wftv.com/news/national-news/ap-top...,2019-10-31,39820.0
1790801,859953698,20190718,20190718,KYOTO,JPN,None,None,JA,"Kyoto, Kyoto, Japan",180,...,18,assault_security_risk,-9.0,3324,409,3181,-6.548347,https://www.tokyoreporter.com/japan/dozens-hur...,2019-07-18,29916.0
1977715,883401413,20191031,20191031,OKINAWA,JPN,CIVILIAN,None,JA,"Okinawa, Okinawa, Japan",1712,...,17,coercion_sanctions_controls,-9.2,3106,390,3106,-3.305755,https://www.wftv.com/news/national-news/ap-top...,2019-10-31,28575.2
1812657,862732641,20190731,20190731,PYONGYANG,PRK,SOUTH KOREA,KOR,KN,"Pyongyang, P'yongyang-si, North Korea",194,...,19,artillery_conflict,-10.0,2711,307,2677,-2.469356,https://www.teletrader.com/n-korea-fires-multi...,2019-07-31,27110.0
1677027,845841815,20190517,20190517,JAPANESE,JPN,None,None,JA,Japan,1712,...,17,coercion_sanctions_controls,-9.2,2736,322,2736,1.262112,https://mynorthwest.com/1386114/i-m-pei-archit...,2019-05-17,25171.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1461537,818808250,20190125,20190125,DEPUTY,None,JAPAN,JPN,JA,"Yokohama, Kanagawa, Japan",173,...,17,arrest_or_detain,-5.0,5,5,5,0.777710,https://www.theolympian.com/news/business/arti...,2019-01-25,25.0
1418591,813426060,20190101,20190101,JAPAN,JPN,None,None,JA,Japan,173,...,17,arrest_or_detain,-5.0,5,2,5,-6.228303,https://www.autocar.co.uk/car-news/industry/ca...,2019-01-01,25.0
1434712,815333850,20190110,20190110,JAPANESE,JPN,JAPAN,JPN,JA,"Tokyo, Tokyo, Japan",173,...,17,arrest_or_detain,-5.0,5,2,5,-9.953802,https://www.theglobeandmail.com/business/inter...,2019-01-10,25.0
1915151,874872580,20190924,20190924,KYOTO,JPN,UNITED STATES,USA,FR,"Paris, France (general), France",173,...,17,arrest_or_detain,-5.0,5,2,5,4.195804,http://www.hurriyetdailynews.com/ara-guler-pho...,2019-09-24,25.0


In [16]:
events_in(ev, "CHN", year=2022, month=[9, 10, 11])

,GLOBALEVENTID,SQLDATE,DATEADDED,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_CountryCode,ActionGeo_FullName,EventCode,...,EventRootCode,event_class,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL,date,intensity
3317614,1063057295,20220915,20220915,CHINESE,CHN,None,None,CH,"Beijing, Beijing, China",190,...,19,military_force,-10.0,2515,121,2502,-2.551567,https://www.strategypage.com/on_point/20220914...,2022-09-15,25150.0
3381949,1073381659,20221124,20221124,BEIJING,CHN,POLICE,None,CH,"Zhengzhou, Henan, China",190,...,19,military_force,-10.0,1385,116,1385,-5.306557,https://www.timesargus.com/features/health/chi...,2022-11-24,13850.0
3384195,1073812825,20221128,20221128,POLICE,None,SHANGHAI,CHN,CH,"Shanghai, Shanghai, China",175,...,17,repression,-9.0,1327,150,1327,-8.458407,https://www.tribuneindia.com/news/world/anti-x...,2022-11-28,11943.0
3381952,1073381703,20221124,20221124,POLICE,None,BEIJING,CHN,CH,"Beijing, Beijing, China",190,...,19,military_force,-10.0,1125,113,1125,-5.419904,https://www.timesargus.com/features/health/chi...,2022-11-24,11250.0
3384194,1073812808,20221128,20221128,SHANGHAI,CHN,POLICE,None,CH,"Shanghai, Shanghai, China",140,...,14,civil_unrest,-6.5,1531,151,1531,-8.452413,https://www.tribuneindia.com/news/world/anti-x...,2022-11-28,9951.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3324279,1064204794,20220922,20220922,CHINESE,CHN,CHINA,CHN,CH,China,173,...,17,arrest_or_detain,-5.0,5,2,5,-6.151168,https://uk.news.yahoo.com/former-chinese-justi...,2022-09-22,25.0
3383115,1073633235,20221126,20221126,None,None,CHINA,CHN,CH,China,173,...,17,arrest_or_detain,-5.0,5,2,5,-9.543945,https://www.northcentralpa.com/news/woman-wake...,2022-11-26,25.0
3315058,1062622436,20220912,20220912,CHINA,CHN,None,None,CH,"Beijing, Beijing, China",173,...,17,arrest_or_detain,-5.0,5,2,5,-5.927808,https://thechronicle.com.gh/hong-kong-court-se...,2022-09-12,25.0
3328654,1064956061,20220927,20220927,CHICAGO,USA,CHINESE,CHN,CH,China,173,...,17,arrest_or_detain,-5.0,5,2,5,-5.161473,https://www.nakedcapitalism.com/2022/09/links-...,2022-09-27,25.0


In [17]:
events_in(ev, year=range(2017, 2024), top=50)

,GLOBALEVENTID,SQLDATE,DATEADDED,Actor1Name,Actor1CountryCode,Actor2Name,Actor2CountryCode,ActionGeo_CountryCode,ActionGeo_FullName,EventCode,...,EventRootCode,event_class,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL,date,intensity
586765,704165397,20171106,20171106,TEXAS,USA,None,None,US,"Texas, United States",193,...,19,small_arms_conflict,-10.0,28455,1627,23414,-6.084484,http://www.geelongadvertiser.com.au/news/world...,2017-11-06,284550.0
1820721,863625203,20190804,20190804,TEXAS,USA,None,None,US,"El Paso, Texas, United States",193,...,19,small_arms_conflict,-10.0,24738,1503,20910,-6.476222,https://www.nwemail.co.uk/news/national/178145...,2019-08-04,247380.0
556872,694335673,20171002,20171002,UNITED STATES,USA,None,None,US,"Las Vegas, Nevada, United States",193,...,19,small_arms_conflict,-10.0,24132,1467,19495,-6.032438,http://www.lasvegasnow.com/news/update-child-f...,2017-10-02,241320.0
559721,694589228,20171003,20171003,NASHVILLE,USA,None,None,US,"Las Vegas, Nevada, United States",193,...,19,small_arms_conflict,-10.0,21829,1111,16180,-4.843193,http://www.850wftl.com/watch-vigil-held-las-ve...,2017-10-03,218290.0
3798887,1129529526,20230927,20230927,HYUNDAI,KOR,None,None,None,None,190,...,19,military_force,-10.0,20940,80,1620,-1.515536,https://www.centralwesterndaily.com.au/story/8...,2023-09-27,209400.0
38288,618863104,20170120,20170120,UNITED STATES,USA,None,None,US,"Washington, District of Columbia, United States",190,...,19,military_force,-10.0,18882,704,18814,-4.806829,http://www.cbc.ca/news/world/trump-inauguratio...,2017-01-20,188820.0
1126815,777504531,20180807,20180807,CALIFORNIA,USA,PRESIDENT,None,US,"California, United States",190,...,19,military_force,-10.0,18029,1082,16064,-4.759773,https://www.ksbw.com/article/ap-fact-check-tru...,2018-08-07,180290.0
1822895,863768237,20190805,20190805,AMERICAN,USA,None,None,US,"El Paso, Texas, United States",193,...,19,small_arms_conflict,-10.0,16399,1294,13678,-7.301308,https://www.dailycommercial.com/zz/news/201908...,2019-08-05,163990.0
590104,704469120,20171107,20171107,TEXAS,USA,None,None,US,"Texas, United States",193,...,19,small_arms_conflict,-10.0,15144,1109,13523,-5.739935,http://www.kltv.com/story/36771707/the-latest-...,2017-11-07,151440.0
1250731,792202567,20181005,20181005,CHICAGO,USA,CHICAGO,USA,US,"Chicago, Illinois, United States",190,...,19,military_force,-10.0,15072,698,8797,-8.996135,https://www.kveo.com/news/national/the-latest-...,2018-10-05,150720.0


In [18]:
def score_severity_salience(d):                 # current default
    return d["GoldsteinScale"].abs() * d["NumMentions"]

def score_reliability_weighted(d):              # reward corroboration
    import numpy as np
    return d["GoldsteinScale"].abs() * d["NumMentions"] * np.log1p(d["NumSources"])

def score_topical(d, keywords):                 # purely "is it about chips"
    text = (d["Actor1Name"].fillna("") + " " + d["Actor2Name"].fillna("") + " " +
            d["SOURCEURL"].fillna("")).str.lower()
    return text.apply(lambda t: sum(k in t for k in keywords)).astype(float)

# --- one selector that takes whichever scorer + policy you want ---
def select_events(d, scorer=score_severity_salience, policy="all", k=10, thresh=None,
                  group=("Actor1CountryCode", "ym")):
    d = d.copy()
    d["score"] = scorer(d)
    if policy == "all":
        return d
    if policy == "threshold":
        return d[d["score"] >= thresh]
    if policy == "topk":
        return d.sort_values("score", ascending=False).groupby(list(group)).head(k)

In [19]:
score_reliability_weighted(china_oc22)

3347946    38525.987133
3357499    21637.829522
3350982    20882.613281
3333339    20838.869162
3352900    19475.639280
3333302    21396.026223
3357498    18659.188873
3352891    17908.121388
3345236    15911.849207
3357510    13474.031209
dtype: float64

In [20]:
score_severity_salience(china_oc22)

3347946    8090.0
3357499    5620.0
3350982    4932.0
3333339    4905.0
3352900    4860.0
3333302    4842.0
3357498    4820.0
3352891    4290.0
3345236    4180.0
3357510    3760.0
dtype: float64

In [21]:
score_topical(china_oc22, ["chip", "silicon", "silicon chip", "silicon chip company"])

3347946    0.0
3357499    0.0
3350982    0.0
3333339    0.0
3352900    0.0
3333302    0.0
3357498    0.0
3352891    0.0
3345236    0.0
3357510    0.0
dtype: float64

In [24]:
from country_codes import ISO3_2_M49

cy = monthly_to_annual(cm).reset_index()
cy["reporterCode"] = cy["iso3"].map(ISO3_2_M49)

# sanity check the bridge before merging:
unmapped = cy[cy["reporterCode"].isna()]["iso3"].unique()
print("mapped:", cy["reporterCode"].notna().sum(), "/", len(cy))
print("unmapped iso3 codes:", unmapped)

mapped: 1265 / 1345
unmapped iso3 codes: ['AFR' 'ASA' 'CRB' 'LAM' 'LIE' 'MCO' 'MEA' 'NMR' 'PGS' 'SAS' 'SEA' 'TMP'
 'TWN' 'WAF' 'WST']


In [23]:
ISO3_2_M49_FIX = {**ISO3_2_M49, "TWN": "490"}   # Taiwan -> Comtrade's 490
cy["reporterCode"] = cy["iso3"].map(ISO3_2_M49_FIX)
cy = cy.dropna(subset=["reporterCode"])          # drop the region/microstate rows
print("kept:", len(cy), "| countries:", cy["reporterCode"].nunique())

kept: 1272 | countries: 197


In [25]:
CRITICAL_ROOTS = ["14", "17", "18", "19", "20"]
ROOT_WEIGHTS   = {"14": 1.0, "17": 1.5, "18": 2.0, "19": 3.0, "20": 4.0}  # fight > protest

def _num(s): return pd.to_numeric(s, errors="coerce")

# ---------- pluggable per-event scorers (each returns a Series) ----------
def score_severity_salience(d):
    """|Goldstein| x NumMentions : type-severity scaled by how loudly it was reported."""
    return _num(d["GoldsteinScale"]).abs() * _num(d["NumMentions"])

def score_reliability_weighted(d):
    """+ reward corroboration across distinct sources."""
    return score_severity_salience(d) * np.log1p(_num(d["NumSources"]))

def score_root_weighted(d):
    """+ weight by event type (fight/mass-violence outweigh protest)."""
    w = d["EventRootCode"].astype(str).map(ROOT_WEIGHTS).fillna(1.0)
    return score_severity_salience(d) * w

# ---------- country-month aggregation WITH normalization ----------
def score_to_country_month(ev, scorer=score_severity_salience,
                           roots=CRITICAL_ROOTS, country_col="Actor1CountryCode"):
    d = ev.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.dropna(subset=["date", country_col])
    if roots is not None:
        d = d[d["EventRootCode"].astype(str).isin(roots)]
    d["ym"]    = d["date"].dt.to_period("M")
    d["score"] = scorer(d)
    d["gold"], d["ment"], d["tone"] = (_num(d["GoldsteinScale"]),
                                       _num(d["NumMentions"]), _num(d["AvgTone"]))
    d["gm"] = d["gold"] * d["ment"].clip(lower=1)          # for coverage-weighted Goldstein

    g = d.groupby([country_col, "ym"])
    out = g.agg(
        n_events   = ("GLOBALEVENTID", "size"),
        score_mean = ("score", "mean"),   # NORMALIZED intensity (volume-free)
        score_max  = ("score", "max"),    # single worst event that month
        score_sum  = ("score", "sum"),    # raw total (volume-biased — use with care)
        tone_mean  = ("tone", "mean"),
        gm_sum     = ("gm", "sum"),
        ment_sum   = ("ment", "sum"),
    )
    out["goldstein_wmean"] = out["gm_sum"] / out["ment_sum"].clip(lower=1)   # Impact-Index style
    return (out.drop(columns=["gm_sum", "ment_sum"]).reset_index()
               .rename(columns={country_col: "iso3"}))

# ---------- annual rollup: structure-preserving + normalized ----------
def score_month_to_annual(cm):
    cm = cm.copy(); cm["year"] = cm["ym"].dt.year
    g = cm.groupby(["iso3", "year"])
    return g.agg(
        events_total    = ("n_events", "sum"),      # raw volume
        score_mean      = ("score_mean", "mean"),   # normalized intensity
        score_max       = ("score_max", "max"),     # worst event of the year = the shock
        score_vol       = ("score_mean", "std"),    # month-to-month volatility
        goldstein_wmean = ("goldstein_wmean", "mean"),
        tone_mean       = ("tone_mean", "mean"),
        active_months   = ("n_events", lambda s: (s > 0).sum()),
    ).reset_index()

In [37]:
from country_codes import ISO3_2_M49
iso2m49 = {**ISO3_2_M49, "TWN": "490"}          # Taiwan -> Comtrade's 490

cy = score_month_to_annual(score_to_country_month(
        pd.read_parquet("gdelt_chip_events_2017_2023.parquet")))
cy["reporterCode"] = cy["iso3"].map(iso2m49)
cy = cy.dropna(subset=["reporterCode"])
cy["reporterCode"] = cy["reporterCode"].astype(int)   # match Comtrade dtype

GDELT_COLS = ["events_total", "score_mean", "score_max", "score_vol",
              "goldstein_wmean", "tone_mean", "active_months"]

def add_gdelt(agg_df):
    """Attach the exporter's GDELT year-vector onto an agg_features frame."""
    out = agg_df.merge(cy[["reporterCode", "year"] + GDELT_COLS],
                       left_on=["reporterCode", "refYear"],
                       right_on=["reporterCode", "year"], how="left")
    out[GDELT_COLS] = out[GDELT_COLS].fillna(0)     # no events that year = zero risk
    return out.drop(columns=["year"])

In [28]:
ev = pd.read_parquet("gdelt_chip_events_2017_2023.parquet")
ev = ev[pd.to_datetime(ev["date"]).dt.year.between(2017, 2023)]
cm = score_to_country_month(ev, scorer=score_severity_salience)
cy = score_month_to_annual(cm)
cy.head()

,iso3,year,events_total,score_mean,score_max,score_vol,goldstein_wmean,tone_mean,active_months
0,ABW,2017,6,63.333333,110.0,NaN,-5.0,-7.958112,1
1,ABW,2018,5,261.000000,774.0,254.558441,-9.0,-9.552747,2
2,ABW,2019,1,40.000000,40.0,NaN,-5.0,-0.731296,1
3,ABW,2020,1,550.000000,550.0,NaN,-5.0,0.649351,1
4,ABW,2023,3,306.250000,510.0,47.729708,-7.5,-7.038944,2


In [33]:
cy.sort_values(by=['year','iso3'], axis=0, ascending=True, inplace=False)

,iso3,year,events_total,score_mean,score_max,score_vol,goldstein_wmean,tone_mean,active_months
0,ABW,2017,6,63.333333,110.0,NaN,-5.000000,-7.958112,1
5,AFG,2017,1224,413.157917,16280.0,168.261722,-9.346548,-5.113536,12
12,AFR,2017,733,331.348768,9920.0,108.828838,-8.172949,-3.981824,12
19,AGO,2017,14,68.410000,120.0,31.983050,-6.723636,-5.679557,5
27,ALB,2017,8,454.375000,1730.0,535.065942,-6.500000,-3.756530,4
...,...,...,...,...,...,...,...,...,...
1316,WST,2023,24,151.138889,540.0,85.686302,-8.901189,-3.788844,9
1323,YEM,2023,258,411.249377,15820.0,347.029984,-9.284004,-7.040553,11
1330,ZAF,2023,55,313.722813,5150.0,553.551961,-7.654463,-5.137885,11
1337,ZMB,2023,15,107.222222,610.0,21.751969,-5.922222,-4.925720,3
